In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

In [ ]:
df_train = pd.read_csv('../../../data/002_processed_data/xy_df.csv')
df_train['dataset_id'] = 'train'
print(f"Training: {len(df_train)}")

df_kaggle = pd.read_csv('../../../data/002_processed_data/x_kaggle_df.csv')
df_kaggle['dataset_id'] = 'kaggle'
df_kaggle['homo'] = np.nan
df_kaggle['lumo'] = np.nan
df_kaggle['stability'] = np.nan
print(f"Kaggle: {len(df_kaggle)}")

df = pd.concat([df_train, df_kaggle], ignore_index=True)
print(f"Total molecules: {len(df)}")

In [ ]:
feature_cols = [c for c in df.columns if c not in ['smiles', 'homo', 'lumo', 'stability', 'dataset_id']]

X_scaled = df[feature_cols].values

print(f"X shape: {X_scaled.shape}")

In [ ]:
k_range = range(2, 21)
inertias = []
silhouettes = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_range, inertias, 'bo-')
axes[0].set_xlabel('Number of clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(k_range, silhouettes, 'ro-')
axes[1].set_xlabel('Number of clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score')

plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(silhouettes)]
print(f"Best k (silhouette): {best_k}")

In [ ]:
k = 10

kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df['cluster_id'] = kmeans.fit_predict(X_scaled)

print(f"Cluster distribution:")
print(df['cluster_id'].value_counts().sort_index())

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=df['cluster_id'], cmap='tab10', alpha=0.6, edgecolor='k', linewidth=0.3)
plt.colorbar(scatter, label='Cluster ID')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('KMeans Clustering (PCA Visualization)')
plt.tight_layout()
plt.show()

In [ ]:
kaggle_clusters = set(df[df['dataset_id'] == 'kaggle']['cluster_id'].unique())
kaggle_clusters = {int(c) for c in kaggle_clusters}
print(f"Clusters containing Kaggle molecules: {sorted(kaggle_clusters)}")

df['is_kaggle_similar'] = df.apply(
    lambda row: row['cluster_id'] in kaggle_clusters if row['dataset_id'] != 'kaggle' else False,
    axis=1
)

print(f"\nTotal training molecules: {len(df[df['dataset_id'] != 'kaggle'])}")
print(f"Training molecules similar to Kaggle: {df['is_kaggle_similar'].sum()}")
print(f"Percentage: {df['is_kaggle_similar'].sum() / len(df[df['dataset_id'] != 'kaggle']) * 100:.1f}%")

In [ ]:
plt.figure(figsize=(10, 8))

mask_not_similar = (df['dataset_id'] != 'kaggle') & (~df['is_kaggle_similar'])
plt.scatter(X_pca[mask_not_similar, 0], X_pca[mask_not_similar, 1], 
            c='gray', alpha=0.4, label='Training (not similar)', edgecolor='k', linewidth=0.2)

mask_similar = df['is_kaggle_similar']
plt.scatter(X_pca[mask_similar, 0], X_pca[mask_similar, 1], 
            c='green', alpha=0.6, label='Training (Kaggle similar)', edgecolor='k', linewidth=0.3)

mask_kaggle = df['dataset_id'] == 'kaggle'
plt.scatter(X_pca[mask_kaggle, 0], X_pca[mask_kaggle, 1], 
            c='red', alpha=0.8, label='Kaggle', edgecolor='k', linewidth=0.3)

plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('Training Molecules Similar to Kaggle')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
kaggle_similarity = df[['smiles', 'cluster_id', 'is_kaggle_similar']].copy()
kaggle_similarity.columns = ['SMILES', 'cluster_id', 'is_kaggle_similar']

print(f"Shape: {kaggle_similarity.shape}")
print(kaggle_similarity.head(10))

In [ ]:
kaggle_similarity.to_csv('kaggle_similarity.csv', index=False)
print("Saved kaggle_similarity.csv")

print(f"\nVerification:")
print(f"Total rows: {len(kaggle_similarity)}")
print(f"Unique SMILES: {kaggle_similarity['SMILES'].nunique()}")
print(f"is_kaggle_similar distribution:")
print(kaggle_similarity['is_kaggle_similar'].value_counts())

In [ ]:
print("="*50)
print("CLUSTERING SUMMARY")
print("="*50)
print(f"Total molecules clustered: {len(df)}")
print(f"Number of clusters: {k}")
print(f"Clusters containing Kaggle molecules: {len(kaggle_clusters)}")
print(f"Training molecules similar to Kaggle: {df['is_kaggle_similar'].sum()} / {len(df[df['dataset_id'] != 'kaggle'])}")
print(f"\nThis information can be used to:")
print("- Give higher weights to similar molecules during training")
print("- Focus validation on similar molecules")
print("- Understand which parts of chemical space Kaggle covers")